Training the first agent

In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import random
import heapq
import itertools
import time
from enum import IntEnum
import os
%pip install stable-baselines3 shimmy
# --- Stable Baselines3 Imports ---
# If this fails, install: pip install stable-baselines3 shimmy
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv


class Actions(IntEnum):
    HOLD = 0
    BUY = 1
    SELL = 2

OBSERVATION_DIMS = 14
INITIAL_CASH = 100_000.0
MAX_STEPS_PER_EPISODE = 2000
MAX_INVENTORY_LIMIT = 100
MAX_CASH_LIMIT = 1_000_000

# Reward Hyperparameters (From Day 3)
INVENTORY_RISK_COEFF = 0.1
DOWNSIDE_PENALTY_MULT = 5.0
TRANSACTION_COST = 1.0


class AdvancedOrderBook:
    def __init__(self):
        self.bids = []  
        self.asks = []  
        self.trades = []
        self.order_id_counter = itertools.count() 

    def submit_order(self, side, qty, price=None, order_type='limit'):
        if order_type == 'market':
            limit_price = float('inf') if side == 'buy' else 0
        else:
            limit_price = price

        remaining_qty = self.match(side, qty, limit_price)

        if remaining_qty > 0 and order_type == 'limit':
            entry_id = next(self.order_id_counter)
            if side == 'buy':
                heapq.heappush(self.bids, [-limit_price, entry_id, remaining_qty])
            else:
                heapq.heappush(self.asks, [limit_price, entry_id, remaining_qty])
            
        return remaining_qty

    def match(self, side, qty, limit_price):
        remaining_qty = qty
        while remaining_qty > 0:
            if side == 'buy':
                if not self.asks: break
                best_price = self.asks[0][0]
                if limit_price < best_price: break
                best_order = self.asks[0]
            else:
                if not self.bids: break
                best_price = -self.bids[0][0] 
                if limit_price > best_price: break
                best_order = self.bids[0]

            trade_qty = min(remaining_qty, best_order[2])
            exec_price = best_order[0] if side == 'buy' else -best_order[0]

            self.trades.append({
                'price': exec_price,
                'qty': trade_qty,
                'timestamp': time.time(),
                'side': side
            })

            remaining_qty -= trade_qty
            best_order[2] -= trade_qty

            if best_order[2] == 0:
                if side == 'buy': heapq.heappop(self.asks)
                else: heapq.heappop(self.bids)
                    
        return remaining_qty

# 3. RL ENVIRONMENT (Risk-Aware)
class TradingEnv(gym.Env):
    metadata = {'render_modes': ['human']}

    def __init__(self):
        super(TradingEnv, self).__init__()
        self.action_space = spaces.Discrete(len(Actions))
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf, 
            shape=(OBSERVATION_DIMS,), dtype=np.float32
        )
        
        self.engine = None
        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.max_portfolio_value = INITIAL_CASH
        self.current_step = 0
        self.mid_price_history = [] 

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.engine = AdvancedOrderBook()
        
        start_price = 100.0
        for i in range(1, 6):
            self.engine.submit_order('buy', 10, start_price - i*0.5, 'limit')
            self.engine.submit_order('sell', 10, start_price + i*0.5, 'limit')

        self.inventory = 0
        self.cash = INITIAL_CASH
        self.portfolio_value = INITIAL_CASH
        self.max_portfolio_value = INITIAL_CASH
        self.current_step = 0
        self.mid_price_history = [start_price] * 10 
        
        return self._get_observation(), {}

    def step(self, action):
        prev_portfolio_value = self.portfolio_value
        
        # Execute
        if action == Actions.BUY and self.cash > 0:
            rem = self.engine.submit_order('buy', 1, order_type='market')
            if rem == 0: 
                fill_price = self.engine.trades[-1]['price']
                self.inventory += 1
                self.cash -= fill_price
        elif action == Actions.SELL:
            rem = self.engine.submit_order('sell', 1, order_type='market')
            if rem == 0: 
                fill_price = self.engine.trades[-1]['price']
                self.inventory -= 1
                self.cash += abs(fill_price)

        # Background Market
        mid = self._get_mid_price()
        shock = np.random.normal(0, 0.5)
        new_fair = mid + shock
        self.engine.submit_order('buy', 10, round(new_fair - 0.5, 2), 'limit')
        self.engine.submit_order('sell', 10, round(new_fair + 0.5, 2), 'limit')

        # Update Portfolio
        mid = self._get_mid_price()
        self.portfolio_value = self.cash + (self.inventory * mid)
        if self.portfolio_value > self.max_portfolio_value:
            self.max_portfolio_value = self.portfolio_value
            
        # Reward Engineering (Day 3 Logic)
        delta_pnl = self.portfolio_value - prev_portfolio_value
        inv_penalty = INVENTORY_RISK_COEFF * abs(self.inventory)
        drawdown_pct = (self.max_portfolio_value - self.portfolio_value) / self.max_portfolio_value
        dd_penalty = drawdown_pct * DOWNSIDE_PENALTY_MULT if drawdown_pct > 0.02 else 0
        trade_cost = TRANSACTION_COST if action != Actions.HOLD else 0
        
        reward = delta_pnl - inv_penalty - dd_penalty - trade_cost
        
        self.current_step += 1
        terminated = self.portfolio_value <= 0 
        truncated = self.current_step >= MAX_STEPS_PER_EPISODE
        
        obs = self._get_observation()
        info = {'inventory': self.inventory, 'pnl': self.portfolio_value - INITIAL_CASH}
        
        return obs, reward, terminated, truncated, info

    def _get_mid_price(self):
        best_bid = -self.engine.bids[0][0] if self.engine.bids else 100.0
        best_ask = self.engine.asks[0][0] if self.engine.asks else 100.0
        mid = (best_bid + best_ask) / 2
        self.mid_price_history.append(mid)
        if len(self.mid_price_history) > 20: self.mid_price_history.pop(0)
        return mid

    def _get_observation(self):
        mid = self._get_mid_price()
        bids = [-x[0] for x in heapq.nsmallest(5, self.engine.bids)] if self.engine.bids else []
        asks = [x[0] for x in heapq.nsmallest(5, self.engine.asks)] if self.engine.asks else []
        while len(bids) < 5: bids.append(mid)
        while len(asks) < 5: asks.append(mid)
        
        norm_bids = [(p - mid)/mid for p in bids]
        norm_asks = [(p - mid)/mid for p in asks]
        norm_inv = self.inventory / MAX_INVENTORY_LIMIT
        norm_cash = self.cash / MAX_CASH_LIMIT
        spread = (asks[0] - bids[0]) / mid
        vol = np.std(self.mid_price_history) / mid if len(self.mid_price_history) > 1 else 0
        
        return np.array(norm_bids + norm_asks + [norm_inv, norm_cash, spread, vol], dtype=np.float32)

# 4. TRAINING SCRIPT

def train_agent():
    print("--- Week 3 Day 4: Training First Agent ---")
    
    # 1. Initialize Environment
    # We wrap it in a DummyVecEnv which is required by Stable-Baselines3
    env = TradingEnv()
    
    # 2. Gym Compliance Check (CRITICAL)
    # This verifies observation/action spaces are valid
    print("Checking Environment Compliance...")
    check_env(env)
    print("Environment is Valid ✅")
    
    # 3. Initialize Agent (PPO)
    # MlpPolicy = Multi-Layer Perceptron (Standard Neural Net)
    # verbose=1 prints training logs (entropy, loss, reward)
    model = PPO("MlpPolicy", env, verbose=1, learning_rate=0.0003, ent_coef=0.01)
    
    print("\nStarting Training (10,000 Timesteps)...")
    print("Look for 'ep_rew_mean' in logs - it should increase!")
    
    # 4. Train
    # This loop manages the Agent <-> Env interaction automatically
    model.learn(total_timesteps=10000)
    
    print("\nTraining Complete.")
    
    # 5. Save Model
    if not os.path.exists("models"): os.makedirs("models")
    model.save("models/ppo_v1_day4")
    print("Model saved to models/ppo_v1_day4.zip")
    
    return model, env

def evaluate_agent(model, env):
    print("\n--- Evaluating Trained Agent ---")
    obs, _ = env.reset()
    
    done = False
    step = 0
    total_reward = 0
    
    # Run 1 Episode
    while not done:
        # Predict action (Deterministic = Best guess, no randomness)
        action, _states = model.predict(obs, deterministic=True)
        
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        step += 1
        
        if step % 100 == 0:
            act_str = ["HOLD", "BUY", "SELL"][action]
            print(f"Step {step}: Action={act_str} | Inv={info['inventory']} | PnL={info['pnl']:.2f}")
            
        done = terminated or truncated
        
    print(f"Evaluation Finished. Total Reward: {total_reward:.2f}")

if __name__ == "__main__":
    # Train
    model, env = train_agent()
    
    # Eval
    evaluate_agent(model, env)

  Using cached stable_baselines3-2.7.1-py3-none-any.whl.metadata (4.8 kB)
  Using cached Shimmy-2.0.0-py3-none-any.whl.metadata (3.5 kB)
Using cached stable_baselines3-2.7.1-py3-none-any.whl (188 kB)
Using cached Shimmy-2.0.0-py3-none-any.whl (30 kB)
Note: you may need to restart the kernel to use updated packages.


--- Week 3 Day 4: Training First Agent ---
Checking Environment Compliance...
Environment is Valid ✅
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

Starting Training (10,000 Timesteps)...
Look for 'ep_rew_mean' in logs - it should increase!
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 2e+03     |
|    ep_rew_mean     | -5.87e+03 |
| time/              |           |
|    fps             | 1725      |
|    iterations      | 1         |
|    time_elapsed    | 1         |
|    total_timesteps | 2048      |
----------------------------------
-------------------------------------------
| rollout/                |               |
|    ep_len_mean          | 2e+03         |
|    ep_rew_mean          | -7.28e+03     |
| time/                   |               |
|    fps                  | 1315          |
|    iterations           | 2             |
|    time_elapsed         | 3             |
|    tota